# 14 AI 辅助量化工作流、风险边界与下一步

## 14.1 本章要解决什么问题

前 13 章已经从真实 ETF 数据、因子、组合、回测、评估一路走到交易信号。本章不再增加新的交易系统，而是把 AI 放在一个更稳妥的位置：它可以帮助阅读代码、规划实验、检查数据和整理复盘，但不能替代本地事实、风控和人工决策。

## 14.2 输入与输出

- 输入：`outputs/results/` 中由前面章节生成的结果文件，例如绩效指标、因子分数、目标权重、订单建议、经典策略汇总。
- 输出：本章会生成闭环检查表、安全提示词模板、AI 建议核验示例、学习路径回顾和下一步路线图。
- 本章产出路径：`outputs/results/chapter14_ai_helper/`。

## 14.3 学完本章你应该能做到

1. 先用代码读取本地结果，再让 AI 在这些事实之上工作。
2. 把 AI 输出拆成可以验证的主张，并用本地文件、字段和指标逐条核对。
3. 写出不会越过交易、收益承诺、数据来源、隐私和复现边界的提示词。
4. 回顾从数据到信号的完整路径，并规划下一阶段练习。


In [1]:
from pathlib import Path
import hashlib
import json
import re
import sys
from datetime import datetime

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "lib").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("请从 pyquant-roadmap 项目根目录或 notebooks/ 目录运行本 notebook。")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.paths import RESULTS_DIR


def rel_path(path: Path) -> str:
    return path.resolve().relative_to(PROJECT_ROOT).as_posix()


CHAPTER_DIR = RESULTS_DIR / "chapter14_ai_helper"
CHAPTER_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("PROJECT_ROOT: .")
print(f"RESULTS_DIR:   {rel_path(RESULTS_DIR)}")
print(f"CHAPTER_DIR:   {rel_path(CHAPTER_DIR)}")


PROJECT_ROOT: .
RESULTS_DIR:   outputs/results
CHAPTER_DIR:   outputs/results/chapter14_ai_helper


## 14.4 先建立本地事实清单

AI 最容易出错的地方，是在没有证据时补全细节。本章的第一步不是写提示词，而是先把已经存在的项目结果文件列出来：哪些文件在、大小是多少、最后修改时间是什么、是否可以作为证据使用。


In [2]:
EXPECTED_ARTIFACTS = [
    ("05 数据质量", "chapter05_data_quality_summary.csv", True, "清洗、对齐后的数据概况"),
    ("06 因子面板", "chapter06_factor_panel.csv", True, "原始因子面板"),
    ("07 因子研究", "factor_ic_summary.csv", True, "IC 检验摘要"),
    ("08 组合构建", "chapter08_latest_target_weights.csv", False, "第 08 章最新目标权重"),
    ("09 回测", "chapter09_strategy_returns.csv", False, "第 09 章回测收益序列"),
    ("10 绩效", "performance_metrics.csv", True, "主线策略绩效指标"),
    ("10 图表", "nav_curve.png", True, "主线策略净值曲线"),
    ("10 图表", "drawdown_curve.png", True, "主线策略回撤曲线"),
    ("10 报告", "quantstats_report.html", False, "QuantStats HTML 报告"),
    ("11 工程化", "experiment_record.json", True, "实验配置与产出记录"),
    ("11 工程化", "factor_scores.csv", True, "工程化流水线因子分数"),
    ("11 工程化", "strategy_returns.csv", True, "工程化流水线收益序列"),
    ("11 工程化", "target_weights.csv", True, "最新目标权重"),
    ("11 工程化", "trade_orders.csv", True, "订单建议表"),
    ("13 经典策略", "classic_strategies/classic_summary.csv", True, "经典策略横向对比"),
]


def rel_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")
    except ValueError:
        return str(path)


def file_digest(path: Path, n: int = 12) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 128), b""):
            digest.update(chunk)
    return digest.hexdigest()[:n]


def artifact_row(stage: str, relative: str, required: bool, description: str) -> dict:
    path = RESULTS_DIR / relative
    stat = path.stat() if path.exists() and path.is_file() else None
    return {
        "stage": stage,
        "artifact": relative,
        "required": required,
        "exists": path.exists(),
        "size_kb": round(stat.st_size / 1024, 1) if stat else np.nan,
        "modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S") if stat else "",
        "fingerprint": file_digest(path),
        "description": description,
        "path": rel_path(path),
    }

artifact_manifest = pd.DataFrame(
    artifact_row(*item) for item in EXPECTED_ARTIFACTS
)
artifact_manifest["status"] = np.where(
    artifact_manifest["exists"],
    "PASS",
    np.where(artifact_manifest["required"], "MISSING", "OPTIONAL_MISSING"),
)

manifest_path = CHAPTER_DIR / "artifact_manifest.csv"
artifact_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

summary = artifact_manifest.groupby("status", as_index=False).size()
print(f"已保存：{rel_path(manifest_path)}")
display(summary)
display(artifact_manifest)


已保存：outputs/results/chapter14_ai_helper/artifact_manifest.csv


,status,size
0,PASS,15


,stage,artifact,required,exists,size_kb,modified,fingerprint,description,path,status
0,05 数据质量,chapter05_data_quality_summary.csv,True,True,0.2000,2026-05-09 00:05:46,c24bf0e8c16a,清洗、对齐后的数据概况,outputs/results/chapter05_data_quality_summary.csv,PASS
1,06 因子面板,chapter06_factor_panel.csv,True,True,214.6000,2026-05-09 00:05:49,b7ad2a8a5081,原始因子面板,outputs/results/chapter06_factor_panel.csv,PASS
2,07 因子研究,factor_ic_summary.csv,True,True,0.9000,2026-05-09 00:06:00,c3241d70e481,IC 检验摘要,outputs/results/factor_ic_summary.csv,PASS
3,08 组合构建,chapter08_latest_target_weights.csv,False,True,0.3000,2026-05-09 00:06:04,dc29721e2931,第 08 章最新目标权重,outputs/results/chapter08_latest_target_weights.csv,PASS
4,09 回测,chapter09_strategy_returns.csv,False,True,54.8000,2026-05-09 00:06:14,be7e911e2d46,第 09 章回测收益序列,outputs/results/chapter09_strategy_returns.csv,PASS
5,10 绩效,performance_metrics.csv,True,True,0.5000,2026-05-09 00:06:25,2858e6a72b47,主线策略绩效指标,outputs/results/performance_metrics.csv,PASS
6,10 图表,nav_curve.png,True,True,69.4000,2026-05-09 00:06:24,df297b5f143b,主线策略净值曲线,outputs/results/nav_curve.png,PASS
7,10 图表,drawdown_curve.png,True,True,57.5000,2026-05-09 00:06:25,fccb06f727f0,主线策略回撤曲线,outputs/results/drawdown_curve.png,PASS
8,10 报告,quantstats_report.html,False,True,727.4000,2026-05-09 00:06:26,006103855e65,QuantStats HTML 报告,outputs/results/quantstats_report.html,PASS
9,11 工程化,experiment_record.json,True,True,1.2000,2026-04-28 09:26:18,e4037412f7b4,实验配置与产出记录,outputs/results/experiment_record.json,PASS


## 14.5 把本地结果整理成闭环检查表

AI 可以帮助发现检查盲区，但检查本身应该由本地代码完成。下面只读取已经落盘的 CSV/JSON，不联网、不调用外部模型，也不假设前面章节一定全部成功。


In [3]:
def read_csv_if_exists(relative: str, **kwargs) -> pd.DataFrame:
    path = RESULTS_DIR / relative
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def read_json_if_exists(relative: str) -> dict:
    path = RESULTS_DIR / relative
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

metrics = read_csv_if_exists("performance_metrics.csv")
factor_scores = read_csv_if_exists("factor_scores.csv", parse_dates=["date"])
factor_ic = read_csv_if_exists("factor_ic_summary.csv")
strategy_returns = read_csv_if_exists("strategy_returns.csv", parse_dates=["date"])
target_weights = read_csv_if_exists("target_weights.csv", parse_dates=["date"])
trade_orders = read_csv_if_exists("trade_orders.csv", parse_dates=["signal_date"])
classic_summary = read_csv_if_exists("classic_strategies/classic_summary.csv")
experiment_record = read_json_if_exists("experiment_record.json")

def metric_value(name: str) -> float | None:
    if metrics.empty or not {"metric", "value"}.issubset(metrics.columns):
        return None
    matched = metrics.loc[metrics["metric"].eq(name), "value"]
    return None if matched.empty else float(matched.iloc[0])


def date_span(df: pd.DataFrame, column: str = "date") -> str:
    if df.empty or column not in df.columns:
        return "缺少日期"
    return f"{df[column].min().date()} 至 {df[column].max().date()}"

facts = {
    "metrics_count": len(metrics),
    "sharpe": metric_value("sharpe"),
    "max_drawdown": metric_value("max_drawdown"),
    "total_return": metric_value("total_return"),
    "factor_score_rows": len(factor_scores),
    "factor_score_assets": int(factor_scores["code"].nunique()) if "code" in factor_scores else 0,
    "factor_score_window": date_span(factor_scores),
    "ic_rows": len(factor_ic),
    "return_rows": len(strategy_returns),
    "return_window": date_span(strategy_returns),
    "latest_weight_date": target_weights["date"].max().date().isoformat() if not target_weights.empty else "缺少",
    "latest_weight_sum": float(target_weights.groupby("date")["target_weight"].sum().iloc[-1]) if not target_weights.empty else np.nan,
    "order_count": len(trade_orders),
    "order_sides": ", ".join(sorted(trade_orders["side"].dropna().astype(str).unique())) if "side" in trade_orders else "缺少",
    "classic_strategy_count": int(classic_summary["strategy"].nunique()) if "strategy" in classic_summary else 0,
    "experiment_top_n": experiment_record.get("top_n"),
    "experiment_cost_bps": experiment_record.get("cost_bps"),
}

pd.DataFrame([facts]).T.rename(columns={0: "local_fact"})


,local_fact
metrics_count,13
sharpe,-0.4567
max_drawdown,-0.3661
total_return,-0.2372
factor_score_rows,2656
factor_score_assets,4
factor_score_window,2021-04-08 至 2023-12-29
ic_rows,6
return_rows,725
return_window,2021-01-04 至 2023-12-29


In [4]:
check_rows = [
    {
        "环节": "项目与路径",
        "本地证据": f"PROJECT_ROOT={rel_path(PROJECT_ROOT)}；RESULTS_DIR={rel_path(RESULTS_DIR)}",
        "状态": "PASS" if RESULTS_DIR.exists() else "MISSING",
        "人工复核点": "确认 notebook 从项目根目录或 notebooks/ 目录运行。",
    },
    {
        "环节": "数据清洗与对齐",
        "本地证据": f"chapter05_data_quality_summary.csv exists={(RESULTS_DIR / 'chapter05_data_quality_summary.csv').exists()}",
        "状态": "PASS" if (RESULTS_DIR / "chapter05_data_quality_summary.csv").exists() else "MISSING",
        "人工复核点": "确认数据来源、复权方式、日期范围和 ETF 池是否符合当前实验。",
    },
    {
        "环节": "因子构建",
        "本地证据": f"factor_scores rows={facts['factor_score_rows']} assets={facts['factor_score_assets']} window={facts['factor_score_window']}",
        "状态": "PASS" if facts["factor_score_rows"] > 0 and facts["factor_score_assets"] > 0 else "MISSING",
        "人工复核点": "检查因子是否只使用当时可得数据，避免未来函数。",
    },
    {
        "环节": "因子检验",
        "本地证据": f"factor_ic_summary rows={facts['ic_rows']}",
        "状态": "PASS" if facts["ic_rows"] > 0 else "MISSING",
        "人工复核点": "IC 只是研究证据，不等同于稳定可交易收益。",
    },
    {
        "环节": "组合权重",
        "本地证据": f"latest_date={facts['latest_weight_date']} weight_sum={facts['latest_weight_sum']:.4f}",
        "状态": "PASS" if np.isfinite(facts["latest_weight_sum"]) and facts["latest_weight_sum"] <= 1.0001 else "WARN",
        "人工复核点": "确认现金、停牌、最小交易单位和调仓频率处理。",
    },
    {
        "环节": "回测收益",
        "本地证据": f"strategy_returns rows={facts['return_rows']} window={facts['return_window']}",
        "状态": "PASS" if facts["return_rows"] > 0 else "MISSING",
        "人工复核点": "确认持仓滞后、交易成本、基准和缺失值处理。",
    },
    {
        "环节": "绩效评估",
        "本地证据": f"metrics={facts['metrics_count']} sharpe={facts['sharpe']:.4f} max_drawdown={facts['max_drawdown']:.4f}",
        "状态": "PASS" if facts["metrics_count"] >= 5 else "MISSING",
        "人工复核点": "不要把单次样本绩效解释成未来收益承诺。",
    },
    {
        "环节": "交易信号",
        "本地证据": f"trade_orders rows={facts['order_count']} sides={facts['order_sides']}",
        "状态": "PASS" if facts["order_count"] > 0 else "WARN",
        "人工复核点": "订单建议只能进入人工复核，不允许由 AI 输出直接连接券商自动下单。",
    },
    {
        "环节": "经典策略对照",
        "本地证据": f"classic_summary strategy_count={facts['classic_strategy_count']}",
        "状态": "PASS" if facts["classic_strategy_count"] >= 3 else "WARN",
        "人工复核点": "横向比较关注结构和风险，不挑单一最高指标当结论。",
    },
]

workflow_checklist = pd.DataFrame(check_rows)
checklist_path = CHAPTER_DIR / "workflow_checklist.csv"
workflow_checklist.to_csv(checklist_path, index=False, encoding="utf-8-sig")

print(f"已保存：{rel_path(checklist_path)}")
display(workflow_checklist)


已保存：outputs/results/chapter14_ai_helper/workflow_checklist.csv


,环节,本地证据,状态,人工复核点
0,项目与路径,PROJECT_ROOT=.；RESULTS_DIR=outputs/results,PASS,确认 notebook 从项目根目录或 notebooks/ 目录运行。
1,数据清洗与对齐,chapter05_data_quality_summary.csv exists=True,PASS,确认数据来源、复权方式、日期范围和 ETF 池是否符合当前实验。
2,因子构建,factor_scores rows=2656 assets=4 window=2021-04-08 至 2023-12-29,PASS,检查因子是否只使用当时可得数据，避免未来函数。
3,因子检验,factor_ic_summary rows=6,PASS,IC 只是研究证据，不等同于稳定可交易收益。
4,组合权重,latest_date=2023-12-29 weight_sum=1.0000,PASS,确认现金、停牌、最小交易单位和调仓频率处理。
5,回测收益,strategy_returns rows=725 window=2021-01-04 至 2023-12-29,PASS,确认持仓滞后、交易成本、基准和缺失值处理。
6,绩效评估,metrics=13 sharpe=-0.4567 max_drawdown=-0.3661,PASS,不要把单次样本绩效解释成未来收益承诺。
7,交易信号,trade_orders rows=3 sides=BUY,PASS,订单建议只能进入人工复核，不允许由 AI 输出直接连接券商自动下单。
8,经典策略对照,classic_summary strategy_count=3,PASS,横向比较关注结构和风险，不挑单一最高指标当结论。


## 14.6 安全的 AI 提示词模板

好的提示词不是让 AI “给一个能赚钱的策略”，而是限定材料、限定任务、限定输出格式，并要求它暴露证据、假设和需要人工复核的部分。下面三个模板分别用于代码审查、实验规划和数据检查。


In [5]:
PROMPT_TEMPLATES = {
    "code_review": """你是量化研究代码审查助手。只能基于我提供的代码、日志和本地结果回答。\n\n任务：审查下面代码是否存在可复现性、数据泄漏、未来函数、日期对齐、交易成本、异常值处理或结果落盘问题。\n\n输出格式：\n1. 结论：通过 / 需要修改 / 信息不足。\n2. 发现列表：每条包含【问题】【证据：具体代码行或输出字段】【影响】【最小修复建议】。\n3. 必须人工复核：列出无法从材料确认的事项。\n\n边界：不承诺收益，不给买卖建议，不生成券商自动下单代码，不要求我提供账号、密钥或隐私数据。\n\n材料：\n{code_or_diff}\n\n本地结果摘要：\n{local_facts}\n""",
    "experiment_planning": """你是量化实验规划助手。请基于当前项目事实，设计一个小步、可复现的实验计划。\n\n当前事实：\n{local_facts}\n\n约束：\n- 一次只改变一个主要变量，例如 lookback、TopN、成本或再平衡频率。\n- 必须保留基准组和失败标准。\n- 必须说明需要保存哪些配置、数据版本、结果文件和随机种子。\n- 只提出研究计划，不把回测结果表述为未来收益。\n\n输出格式：实验问题 / 对照组 / 变量 / 指标 / 复现记录 / 风险与停止条件。\n""",
    "data_sanity_check": """你是量化数据质量检查助手。只能根据我给你的字段摘要、样例行和本地检查结果回答。\n\n请检查：\n1. 日期范围、交易日数量和资产数量是否和实验设定一致。\n2. OHLCV 字段是否缺失、重复、为负或不满足 high >= low。\n3. 收益率、复权方式、数据源、抓取时间是否写清楚。\n4. 哪些结论必须回到本地 CSV 或图表重新确认。\n\n边界：不要编造数据源，不要替我决定实盘交易，不要要求账号、手机号、券商流水、API Key。\n\n字段摘要与样例：\n{data_profile}\n""",
}

def compact_prompt_table(templates: dict[str, str]) -> pd.DataFrame:
    rows = []
    for name, template in templates.items():
        rows.append(
            {
                "template": name,
                "适用场景": {
                    "code_review": "审查 notebook 单元、函数或 diff",
                    "experiment_planning": "规划下一轮小步实验",
                    "data_sanity_check": "检查行情、因子和收益数据质量",
                }[name],
                "关键边界": "限定材料、要求证据、列出人工复核点、拒绝收益承诺和自动下单",
                "占位符": ", ".join(sorted(set(re.findall(r"\{([a-zA-Z0-9_]+)\}", template)))),
            }
        )
    return pd.DataFrame(rows)

prompt_table = compact_prompt_table(PROMPT_TEMPLATES)

prompt_md_lines = ["# Chapter 14 AI Prompt Templates", ""]
for name, template in PROMPT_TEMPLATES.items():
    prompt_md_lines.extend([f"## {name}", "", "```text", template.strip(), "```", ""])

prompt_table_path = CHAPTER_DIR / "prompt_templates.csv"
prompt_md_path = CHAPTER_DIR / "prompt_templates.md"
prompt_table.to_csv(prompt_table_path, index=False, encoding="utf-8-sig")
prompt_md_path.write_text("\n".join(prompt_md_lines), encoding="utf-8")

print(f"已保存：{rel_path(prompt_table_path)}")
print(f"已保存：{rel_path(prompt_md_path)}")
display(prompt_table)
print(PROMPT_TEMPLATES["code_review"][:700])


已保存：outputs/results/chapter14_ai_helper/prompt_templates.csv
已保存：outputs/results/chapter14_ai_helper/prompt_templates.md


,template,适用场景,关键边界,占位符
0,code_review,审查 notebook 单元、函数或 diff,限定材料、要求证据、列出人工复核点、拒绝收益承诺和自动下单,"code_or_diff, local_facts"
1,experiment_planning,规划下一轮小步实验,限定材料、要求证据、列出人工复核点、拒绝收益承诺和自动下单,local_facts
2,data_sanity_check,检查行情、因子和收益数据质量,限定材料、要求证据、列出人工复核点、拒绝收益承诺和自动下单,data_profile


你是量化研究代码审查助手。只能基于我提供的代码、日志和本地结果回答。

任务：审查下面代码是否存在可复现性、数据泄漏、未来函数、日期对齐、交易成本、异常值处理或结果落盘问题。

输出格式：
1. 结论：通过 / 需要修改 / 信息不足。
2. 发现列表：每条包含【问题】【证据：具体代码行或输出字段】【影响】【最小修复建议】。
3. 必须人工复核：列出无法从材料确认的事项。

边界：不承诺收益，不给买卖建议，不生成券商自动下单代码，不要求我提供账号、密钥或隐私数据。

材料：
{code_or_diff}

本地结果摘要：
{local_facts}



## 14.7 把 AI 建议拆成可核验主张

AI 输出不能直接进入研究结论。更稳妥的做法是把一句建议拆成若干主张，再用本地表格核对。下面用几个常见 AI 建议演示：有的能被本地事实支持，有的会被本地事实反驳，有的即使事实正确也越过了交易边界。


In [6]:
def latest_top_score_codes(scores: pd.DataFrame, n: int) -> set[str]:
    if scores.empty or not {"date", "code", "score"}.issubset(scores.columns):
        return set()
    latest_date = scores["date"].max()
    latest = scores.loc[scores["date"].eq(latest_date)].sort_values("score", ascending=False)
    return set(latest.head(n)["code"].astype(str))


def latest_target_codes(weights: pd.DataFrame) -> set[str]:
    if weights.empty or not {"date", "code", "target_weight"}.issubset(weights.columns):
        return set()
    latest_date = weights["date"].max()
    latest = weights.loc[weights["date"].eq(latest_date) & weights["target_weight"].gt(0)]
    return set(latest["code"].astype(str))

score_top_codes = latest_top_score_codes(factor_scores, n=3)
target_codes = latest_target_codes(target_weights)
actual_sharpe = facts["sharpe"]
orders_have_buy = "BUY" in facts["order_sides"].split(", ")

verification_rows = [
    {
        "AI 建议或主张": "最新目标权重应该来自 factor_scores.csv 最新日期的 score Top3。",
        "本地核验": f"score_top3={sorted(score_top_codes)} target_codes={sorted(target_codes)}",
        "判断": "SUPPORTED" if score_top_codes == target_codes and target_codes else "NEEDS_REVIEW",
        "下一步": "如果不一致，检查排序方向、调仓日期和 TopN 配置。",
    },
    {
        "AI 建议或主张": "这个主线策略的 Sharpe 大约是 1.2。",
        "本地核验": f"performance_metrics.csv 中 sharpe={actual_sharpe:.4f}",
        "判断": "REJECTED" if actual_sharpe is not None and abs(actual_sharpe - 1.2) > 0.05 else "SUPPORTED",
        "下一步": "要求 AI 引用具体指标文件；不要接受没有证据的绩效数字。",
    },
    {
        "AI 建议或主张": "trade_orders.csv 有 BUY 单，所以可以把这些订单自动提交到券商。",
        "本地核验": f"trade_orders 包含 BUY={orders_have_buy}；但本项目只生成订单建议表。",
        "判断": "BOUNDARY_BLOCKED",
        "下一步": "保留人工复核、风控检查和真实账户隔离；本 notebook 不写 broker 自动化代码。",
    },
    {
        "AI 建议或主张": "回测覆盖 2021 到 2023 年，可以继续比较经典策略结果。",
        "本地核验": f"strategy_returns window={facts['return_window']}；classic_strategy_count={facts['classic_strategy_count']}",
        "判断": "SUPPORTED" if facts["return_rows"] > 0 and facts["classic_strategy_count"] >= 3 else "NEEDS_REVIEW",
        "下一步": "比较前先确认同一资产池、同一成本假设、同一基准。",
    },
]

verification_table = pd.DataFrame(verification_rows)
verification_path = CHAPTER_DIR / "ai_suggestion_verification.csv"
verification_table.to_csv(verification_path, index=False, encoding="utf-8-sig")

print(f"已保存：{rel_path(verification_path)}")
display(verification_table)


已保存：outputs/results/chapter14_ai_helper/ai_suggestion_verification.csv


,AI 建议或主张,本地核验,判断,下一步
0,最新目标权重应该来自 factor_scores.csv 最新日期的 score Top3。,"score_top3=['510300', '510500', '512100'] target_codes=['510300', '510500', '512100']",SUPPORTED,如果不一致，检查排序方向、调仓日期和 TopN 配置。
1,这个主线策略的 Sharpe 大约是 1.2。,performance_metrics.csv 中 sharpe=-0.4567,REJECTED,要求 AI 引用具体指标文件；不要接受没有证据的绩效数字。
2,trade_orders.csv 有 BUY 单，所以可以把这些订单自动提交到券商。,trade_orders 包含 BUY=True；但本项目只生成订单建议表。,BOUNDARY_BLOCKED,保留人工复核、风控检查和真实账户隔离；本 notebook 不写 broker 自动化代码。
3,回测覆盖 2021 到 2023 年，可以继续比较经典策略结果。,strategy_returns window=2021-01-04 至 2023-12-29；classic_strategy_count=3,SUPPORTED,比较前先确认同一资产池、同一成本假设、同一基准。


## 14.8 风险边界：AI 可以加速，但不能越权

以下边界比提示词更重要。只要 AI 输出触碰这些边界，即使它的语气很肯定，也应该停下来回到本地事实、人工复核和风险控制流程。


In [7]:
risk_boundaries = pd.DataFrame(
    [
        {
            "边界": "券商与实盘自动化",
            "AI 可以做": "解释订单建议字段、生成复核清单、指出代码风险。",
            "必须人工处理": "不从 AI 输出直接连接券商 API，不自动提交、撤单或改仓。",
        },
        {
            "边界": "收益与胜率表述",
            "AI 可以做": "复述本地回测指标和样本区间。",
            "必须人工处理": "不写保证收益、稳定盈利、确定跑赢等表述。",
        },
        {
            "边界": "数据来源与时间",
            "AI 可以做": "根据字段摘要列出待核验问题。",
            "必须人工处理": "确认 AKShare 来源、复权方式、抓取时间、交易日历和缓存版本。",
        },
        {
            "边界": "隐私与密钥",
            "AI 可以做": "基于匿名化样例、字段名、统计摘要工作。",
            "必须人工处理": "不提交账号、手机号、券商流水、API Key、完整本地私有路径或身份信息。",
        },
        {
            "边界": "生成代码",
            "AI 可以做": "给出小段函数、测试思路和审查清单。",
            "必须人工处理": "本地运行测试、检查 diff、确认没有覆盖已有缓存和他人修改。",
        },
        {
            "边界": "策略结论",
            "AI 可以做": "组织证据、比较假设、提示遗漏变量。",
            "必须人工处理": "不把单次回测当成结论；新增实验要有对照组和失败标准。",
        },
    ]
)

risk_path = CHAPTER_DIR / "risk_boundaries.csv"
risk_boundaries.to_csv(risk_path, index=False, encoding="utf-8-sig")

print(f"已保存：{rel_path(risk_path)}")
display(risk_boundaries)


已保存：outputs/results/chapter14_ai_helper/risk_boundaries.csv


,边界,AI 可以做,必须人工处理
0,券商与实盘自动化,解释订单建议字段、生成复核清单、指出代码风险。,不从 AI 输出直接连接券商 API，不自动提交、撤单或改仓。
1,收益与胜率表述,复述本地回测指标和样本区间。,不写保证收益、稳定盈利、确定跑赢等表述。
2,数据来源与时间,根据字段摘要列出待核验问题。,确认 AKShare 来源、复权方式、抓取时间、交易日历和缓存版本。
3,隐私与密钥,基于匿名化样例、字段名、统计摘要工作。,不提交账号、手机号、券商流水、API Key、完整本地私有路径或身份信息。
4,生成代码,给出小段函数、测试思路和审查清单。,本地运行测试、检查 diff、确认没有覆盖已有缓存和他人修改。
5,策略结论,组织证据、比较假设、提示遗漏变量。,不把单次回测当成结论；新增实验要有对照组和失败标准。


## 14.9 练习：给 AI 输出加一层本地核验

练习目标：不要直接相信下面这句 AI 输出，而是写出需要核验的本地文件、字段和判断规则。

> “可以把 TopN 从 3 提高到 4，因为资产池有 4 只 ETF，这样一定能降低回撤。”

先完成下一格中的 scaffold，再对照答案。关键点是：TopN=4 是否可行是一回事，“一定降低回撤”是另一回事。


In [8]:
exercise_claim = "可以把 TopN 从 3 提高到 4，因为资产池有 4 只 ETF，这样一定能降低回撤。"

exercise_scaffold = pd.DataFrame(
    [
        {"需要核验": "资产池数量", "本地文件或字段": "factor_scores.csv: code", "判断规则": "最新日期可选资产数量是否 >= 4"},
        {"需要核验": "当前 TopN 配置", "本地文件或字段": "experiment_record.json: top_n", "判断规则": "确认当前参数是不是 3"},
        {"需要核验": "降低回撤", "本地文件或字段": "新的对照实验 + performance_metrics.csv: max_drawdown", "判断规则": "必须运行 TopN=4 对照组后比较，不能由 AI 直接断言"},
        {"需要核验": "交易影响", "本地文件或字段": "target_weights.csv / trade_orders.csv", "判断规则": "检查换手、订单数量、最小交易单位和成本变化"},
    ]
)

print(exercise_claim)
display(exercise_scaffold)


可以把 TopN 从 3 提高到 4，因为资产池有 4 只 ETF，这样一定能降低回撤。


,需要核验,本地文件或字段,判断规则
0,资产池数量,factor_scores.csv: code,最新日期可选资产数量是否 >= 4
1,当前 TopN 配置,experiment_record.json: top_n,确认当前参数是不是 3
2,降低回撤,新的对照实验 + performance_metrics.csv: max_drawdown,必须运行 TopN=4 对照组后比较，不能由 AI 直接断言
3,交易影响,target_weights.csv / trade_orders.csv,检查换手、订单数量、最小交易单位和成本变化


In [9]:
latest_asset_count = (
    factor_scores.loc[factor_scores["date"].eq(factor_scores["date"].max()), "code"].nunique()
    if not factor_scores.empty and {"date", "code"}.issubset(factor_scores.columns)
    else 0
)
current_top_n = facts["experiment_top_n"]

exercise_answer = pd.DataFrame(
    [
        {
            "子主张": "资产池有 4 只 ETF，因此 TopN=4 在技术上可设置。",
            "证据": f"最新因子日期可选资产数={latest_asset_count}；experiment_record.top_n={current_top_n}",
            "判断": "SUPPORTED" if latest_asset_count >= 4 else "NEEDS_REVIEW",
        },
        {
            "子主张": "TopN=4 一定降低回撤。",
            "证据": "当前没有 TopN=4 对照实验结果。",
            "判断": "REJECTED_AS_UNVERIFIED",
        },
        {
            "子主张": "下一步应该运行小步对照实验。",
            "证据": "只改变 TopN，保留成本、样本区间、因子权重、再平衡规则，再比较 max_drawdown、turnover 和收益指标。",
            "判断": "ACTIONABLE",
        },
    ]
)

exercise_path = CHAPTER_DIR / "exercise_answer.csv"
exercise_answer.to_csv(exercise_path, index=False, encoding="utf-8-sig")
print(f"已保存：{rel_path(exercise_path)}")
display(exercise_answer)


已保存：outputs/results/chapter14_ai_helper/exercise_answer.csv


,子主张,证据,判断
0,资产池有 4 只 ETF，因此 TopN=4 在技术上可设置。,最新因子日期可选资产数=4；experiment_record.top_n=3,SUPPORTED
1,TopN=4 一定降低回撤。,当前没有 TopN=4 对照实验结果。,REJECTED_AS_UNVERIFIED
2,下一步应该运行小步对照实验。,只改变 TopN，保留成本、样本区间、因子权重、再平衡规则，再比较 max_drawdown、turnover 和收益指标。,ACTIONABLE


## 14.10 从数据到信号的学习路径回顾

这一轮学习主线不是“找到神奇参数”，而是建立可复现的研究闭环。下面的回顾表把每一步的输入、输出和适合让 AI 辅助的问题放在一起。AI 的角色始终是研究助手，不是交易代理。


In [10]:
learning_path = pd.DataFrame(
    [
        ("01-02", "环境与主线", "确认项目结构、依赖和第一轮运行", "02_first_run_healthcheck.csv", "帮我解释环境检查失败项，但只基于日志回答。"),
        ("03", "pandas / numpy", "掌握表格、时间序列和向量化计算", "notebook 输出", "帮我把循环改成向量化，并指出结果是否等价。"),
        ("04-05", "数据获取与清洗", "获取真实 ETF 数据，统一字段，清洗对齐", "chapter05_data_quality_summary.csv", "根据字段摘要列出数据质量复核清单。"),
        ("06-07", "因子与检验", "构造动量、波动、均线差，并做 IC 检查", "factor_ic_summary.csv", "审查因子是否可能使用未来数据。"),
        ("08", "组合构建", "把分数转成 TopN 等权目标权重", "chapter08_target_weight_matrix.csv", "检查权重和、调仓日和空仓处理是否一致。"),
        ("09-10", "回测与评估", "加入持仓滞后、成本、净值、回撤、基准对比", "performance_metrics.csv", "帮我把绩效结论改写为有样本边界的复盘。"),
        ("11", "工程化流水线", "用配置复现实验并输出交易信号", "experiment_record.json / trade_orders.csv", "设计下一轮只改一个变量的实验计划。"),
        ("12-13", "策略地图与经典策略", "理解策略结构，比较双均线、布林带、横截面动量", "classic_summary.csv", "基于同一成本和样本，整理策略差异和风险。"),
        ("14", "AI 辅助与边界", "用本地事实约束 AI，建立复核和下一步路线", "chapter14_ai_helper/*", "把 AI 输出拆成可核验主张。"),
    ],
    columns=["章节", "主题", "你已经完成的能力", "代表产出", "适合的 AI 辅助问题"],
)

learning_path_path = CHAPTER_DIR / "learning_path_recap.csv"
learning_path.to_csv(learning_path_path, index=False, encoding="utf-8-sig")
print(f"已保存：{rel_path(learning_path_path)}")
display(learning_path)


已保存：outputs/results/chapter14_ai_helper/learning_path_recap.csv


,章节,主题,你已经完成的能力,代表产出,适合的 AI 辅助问题
0,01-02,环境与主线,确认项目结构、依赖和第一轮运行,02_first_run_healthcheck.csv,帮我解释环境检查失败项，但只基于日志回答。
1,03,pandas / numpy,掌握表格、时间序列和向量化计算,notebook 输出,帮我把循环改成向量化，并指出结果是否等价。
2,04-05,数据获取与清洗,获取真实 ETF 数据，统一字段，清洗对齐,chapter05_data_quality_summary.csv,根据字段摘要列出数据质量复核清单。
3,06-07,因子与检验,构造动量、波动、均线差，并做 IC 检查,factor_ic_summary.csv,审查因子是否可能使用未来数据。
4,08,组合构建,把分数转成 TopN 等权目标权重,chapter08_target_weight_matrix.csv,检查权重和、调仓日和空仓处理是否一致。
5,09-10,回测与评估,加入持仓滞后、成本、净值、回撤、基准对比,performance_metrics.csv,帮我把绩效结论改写为有样本边界的复盘。
6,11,工程化流水线,用配置复现实验并输出交易信号,experiment_record.json / trade_orders.csv,设计下一轮只改一个变量的实验计划。
7,12-13,策略地图与经典策略,理解策略结构，比较双均线、布林带、横截面动量,classic_summary.csv,基于同一成本和样本，整理策略差异和风险。
8,14,AI 辅助与边界,用本地事实约束 AI，建立复核和下一步路线,chapter14_ai_helper/*,把 AI 输出拆成可核验主张。


## 14.11 下一步路线图

下一阶段不要急着扩大系统。更稳妥的路线是先把复现、数据质量、实验记录和风险解释做扎实，再逐步增加策略复杂度。


In [11]:
roadmap = pd.DataFrame(
    [
        {
            "阶段": "第 1 步：复现实验",
            "练习": "重新运行主线流水线，确认 outputs/results 中关键文件能稳定生成。",
            "保留证据": "artifact_manifest.csv、experiment_record.json、performance_metrics.csv",
            "停止条件": "文件缺失、日期范围变化无法解释、指标和上次大幅不一致。",
        },
        {
            "阶段": "第 2 步：做一个小改动",
            "练习": "只改变 TopN、lookback、成本或调仓频率中的一个变量。",
            "保留证据": "配置文件、对照组指标、净值图、回撤图、订单变化。",
            "停止条件": "没有基准组，或同时改变多个变量导致无法归因。",
        },
        {
            "阶段": "第 3 步：加强数据检查",
            "练习": "为数据缓存增加日期、资产、缺失、重复、OHLCV 合法性检查。",
            "保留证据": "数据质量摘要、异常样例、处理规则。",
            "停止条件": "数据来源、复权方式或缓存时间说不清楚。",
        },
        {
            "阶段": "第 4 步：扩展策略地图",
            "练习": "新增一个策略类型，但先写清楚假设、信号、组合、成本和失败标准。",
            "保留证据": "策略说明、最小实现、对照实验、风险解释。",
            "停止条件": "只有回测结果，没有结构解释和风险解释。",
        },
        {
            "阶段": "第 5 步：建立研究日志",
            "练习": "每次实验记录配置、数据版本、指标、图表、结论和下一步。",
            "保留证据": "experiment_record.json、Markdown 复盘、关键 CSV 指纹。",
            "停止条件": "无法从结果反推出当时使用的参数和数据。",
        },
    ]
)

roadmap_path = CHAPTER_DIR / "next_step_roadmap.csv"
roadmap.to_csv(roadmap_path, index=False, encoding="utf-8-sig")
print(f"已保存：{rel_path(roadmap_path)}")
display(roadmap)


已保存：outputs/results/chapter14_ai_helper/next_step_roadmap.csv


,阶段,练习,保留证据,停止条件
0,第 1 步：复现实验,重新运行主线流水线，确认 outputs/results 中关键文件能稳定生成。,artifact_manifest.csv、experiment_record.json、performance_metrics.csv,文件缺失、日期范围变化无法解释、指标和上次大幅不一致。
1,第 2 步：做一个小改动,只改变 TopN、lookback、成本或调仓频率中的一个变量。,配置文件、对照组指标、净值图、回撤图、订单变化。,没有基准组，或同时改变多个变量导致无法归因。
2,第 3 步：加强数据检查,为数据缓存增加日期、资产、缺失、重复、OHLCV 合法性检查。,数据质量摘要、异常样例、处理规则。,数据来源、复权方式或缓存时间说不清楚。
3,第 4 步：扩展策略地图,新增一个策略类型，但先写清楚假设、信号、组合、成本和失败标准。,策略说明、最小实现、对照实验、风险解释。,只有回测结果，没有结构解释和风险解释。
4,第 5 步：建立研究日志,每次实验记录配置、数据版本、指标、图表、结论和下一步。,experiment_record.json、Markdown 复盘、关键 CSV 指纹。,无法从结果反推出当时使用的参数和数据。


## 14.12 小结

本章的核心做法是：先取本地事实，再提问；先拆分主张，再验证；先设边界，再让 AI 加速。

你现在已经具备一个完整的个人量化研究闭环：真实数据获取、字段标准化、清洗对齐、因子构建、因子检验、组合构建、回测、绩效评估、工程化运行、交易信号、策略对照和 AI 辅助复核。下一步的重点不是让 AI 直接给答案，而是让它帮助你更快地发现盲区，同时用本地代码和结果文件守住复现、风险和隐私边界。
